# LeetCode 341 — Flatten Nested List Iterator bằng Python

Link đề: https://leetcode.com/problems/flatten-nested-list-iterator/ · Độ khó: **Medium** · Chủ đề: Stack, Tree, Depth-First Search, Design, Queue, Iterator

Skeleton LeetCode (giao diện `NestedInteger` do LeetCode cung cấp, ta **không** tự cài trong bài nộp):

```python
# """
# This is the interface that allows for creating nested lists.
# You should not implement it, or speculate about its implementation
# """
# class NestedInteger:
#    def isInteger(self) -> bool:
#        """@return True if this NestedInteger holds a single integer, rather than a nested list."""
#    def getInteger(self) -> int:
#        """@return the single integer that this NestedInteger holds, if it holds a single integer
#        The result is undefined if this NestedInteger holds a nested list"""
#    def getList(self) -> [NestedInteger]:
#        """@return the nested list that this NestedInteger holds, if it holds a nested list
#        The result is undefined if this NestedInteger holds a single integer"""

class NestedIterator:
    def __init__(self, nestedList: [NestedInteger]):

    def next(self) -> int:

    def hasNext(self) -> bool:

# Your NestedIterator object will be instantiated and called as such:
# i, v = NestedIterator(nestedList), []
# while i.hasNext(): v.append(i.next())
```

Ghi chú: đề, ví dụ và constraints lấy từ file đề có sẵn (bài này không phải Premium). Trong notebook ta **tự dựng `NestedInteger` giả lập** (đúng ba phương thức `isInteger`, `getInteger`, `getList`) và một khung test iterator.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho danh sách lồng nhau (mỗi phần tử là **số nguyên** hoặc **danh sách** cũng chứa số nguyên/danh sách), cài một **iterator** `next()`, `hasNext()` trả các số nguyên theo thứ tự **duyệt sâu trước, trái sang phải** (như "dàn phẳng").
2. Hiểu giao diện bài **thiết kế**: lớp `NestedInteger` do hệ thống cung cấp; bạn chỉ dùng `isInteger()`, `getInteger()`, `getList()`.
3. Đi từ **dàn phẳng ngay trong constructor bằng đệ quy** → **dàn phẳng bằng ngăn xếp (không đệ quy)** → **iterator lười dùng ngăn xếp các `NestedInteger`** (cách nộp thông dụng) → **ngăn xếp các iterator con** (bộ nhớ theo độ sâu) → **generator `yield from` + nhìn trước**.
4. Hiểu **vì sao `hasNext()` mới là hàm làm việc**: nó phải "mở" các danh sách (kể cả danh sách **rỗng**) cho đến khi đỉnh là một số nguyên thật; và `hasNext()` phải **không tiêu thụ** phần tử (gọi nhiều lần vẫn an toàn).
5. Hiểu khác biệt **dàn phẳng sớm (eager)** và **lười (lazy)**: chi phí constructor, bộ nhớ, và hành vi khi cấu trúc rất sâu.
6. Biết kiểm tra bài thiết kế bằng khung `collect` (đúng như giả mã của đề) và khung **gọi xen kẽ ngẫu nhiên** `hasNext()` / `next()`.
7. Tránh các lỗi kinh điển: `hasNext` chỉ kiểm tra ngăn xếp khác rỗng, đẩy danh sách không đảo ngược, `hasNext` làm mất phần tử, chỉ dàn phẳng một tầng, và đệ quy quá sâu.

## 2. Đề bài & ví dụ

Cho một danh sách số nguyên lồng nhau `nestedList`. Mỗi phần tử là một số nguyên hoặc một danh sách mà các phần tử của nó cũng có thể là số nguyên hoặc danh sách khác. Hãy cài một **iterator** để dàn phẳng nó.

Cài lớp `NestedIterator`:

- `NestedIterator(List<NestedInteger> nestedList)` khởi tạo iterator với `nestedList`.
- `int next()` trả về số nguyên kế tiếp.
- `boolean hasNext()` trả `true` nếu còn số nguyên, ngược lại `false`.

Mã sẽ được kiểm tra bằng giả mã:

```
khởi tạo iterator với nestedList
res = []
while iterator.hasNext()
    thêm iterator.next() vào cuối res
return res
```

Nếu `res` khớp danh sách dàn phẳng mong đợi thì được chấp nhận.

**Ví dụ 1:** `nestedList = [[1,1],2,[1,1]]` → `[1,1,2,1,1]`.

**Ví dụ 2:** `nestedList = [1,[4,[6]]]` → `[1,4,6]`.

**Ràng buộc:** `1 <= nestedList.length <= 500`; giá trị số nguyên trong `[-10^6, 10^6]`.

In [1]:
import re
import random
import sys
import time
from typing import List


class NestedInteger:
    """Giả lập giao diện NestedInteger của LeetCode: chứa HOẶC một số nguyên HOẶC một danh sách NestedInteger."""

    calls = 0                                     # đếm số lần gọi giao diện (để minh họa tính "lười")

    def __init__(self, value=None):
        if isinstance(value, list):
            self._list = value
            self._int = None
        else:
            self._list = None
            self._int = value

    def isInteger(self) -> bool:
        NestedInteger.calls += 1
        return self._list is None

    def getInteger(self):
        NestedInteger.calls += 1
        return self._int

    def getList(self):
        NestedInteger.calls += 1
        return self._list

    def __repr__(self):
        return repr(self._int) if self._list is None else repr(self._list)


def to_nested(obj):
    """Số nguyên hoặc danh sách Python lồng nhau -> NestedInteger."""
    if isinstance(obj, list):
        return NestedInteger([to_nested(x) for x in obj])
    return NestedInteger(obj)


def to_nested_list(py_list):
    """Danh sách Python lồng nhau -> list[NestedInteger] (đúng kiểu tham số của constructor)."""
    return [to_nested(x) for x in py_list]


def expected_flatten(py_list):
    """Oracle độc lập: lấy các số nguyên theo thứ tự xuất hiện trong chuỗi biểu diễn (regex), không đệ quy trên cấu trúc."""
    return [int(tok) for tok in re.findall(r"-?\d+", repr(py_list))]


examples = [
    ([[1, 1], 2, [1, 1]], [1, 1, 2, 1, 1]),
    ([1, [4, [6]]], [1, 4, 6]),
]
for nested, expected in examples:
    ni = to_nested_list(nested)
    print(nested, "| biểu diễn NestedInteger:", ni, "| dàn phẳng kỳ vọng:", expected, "| oracle regex:", expected_flatten(nested))

[[1, 1], 2, [1, 1]] | biểu diễn NestedInteger: [[1, 1], 2, [1, 1]] | dàn phẳng kỳ vọng: [1, 1, 2, 1, 1] | oracle regex: [1, 1, 2, 1, 1]
[1, [4, [6]]] | biểu diễn NestedInteger: [1, [4, [6]]] | dàn phẳng kỳ vọng: [1, 4, 6] | oracle regex: [1, 4, 6]


## 3. Kiến thức nền

### 3.1. Giao diện `NestedInteger`

Mỗi `NestedInteger` là **một trong hai loại**: chứa **một số nguyên** (`isInteger() == True`, lấy bằng `getInteger()`) hoặc chứa **một danh sách** các `NestedInteger` khác (`isInteger() == False`, lấy bằng `getList()`). Cấu trúc này giống **cây**: số nguyên là **lá**, danh sách là **nút trong**; dàn phẳng = **duyệt cây theo thứ tự trước (DFS)** và gom các lá.
#### Code demo

In [2]:
x = to_nested([1, [4, [6]]])
print("danh sách ngoài cùng isInteger:", x.isInteger(), "| getList():", x.getList())
y = x.getList()[0]
print("phần tử đầu isInteger:", y.isInteger(), "| getInteger():", y.getInteger())

danh sách ngoài cùng isInteger: False | getList(): [1, [4, [6]]]
phần tử đầu isInteger: True | getInteger(): 1


### 3.2. Iterator và giao thức `hasNext()` / `next()`

Iterator trả các phần tử **từng cái một theo yêu cầu**. Hợp đồng: `hasNext()` cho biết còn phần tử không **mà không làm mất phần tử**; `next()` trả phần tử kế tiếp. Người dùng có thể gọi `hasNext()` nhiều lần liên tiếp, hoặc (trong test của đề) luôn `hasNext()` trước rồi `next()`.
#### Code demo

In [3]:
class CountDown:
    def __init__(self, n):
        self.n = n

    def hasNext(self):
        return self.n > 0

    def next(self):
        self.n -= 1
        return self.n + 1


it = CountDown(3)
print(it.hasNext(), it.hasNext(), it.next(), it.next(), it.next(), it.hasNext())     # hasNext không tiêu thụ

True True 3 2 1 False


### 3.3. Ngăn xếp (stack) và thứ tự đảo ngược

Ngăn xếp lấy ra phần tử **vào sau cùng**. Muốn xử lý danh sách theo thứ tự trái → phải bằng ngăn xếp, ta đẩy các phần tử vào **theo thứ tự đảo ngược** (phần tử đầu nằm trên đỉnh). Mỗi lần gặp một danh sách con ta thay nó bằng các phần tử của nó, **đảo ngược**, ngay trên đỉnh.
#### Code demo

In [4]:
stack = [3, 2, 1][::-1]            # đỉnh là phần tử đầu của danh sách gốc [3, 2, 1]
print("lấy ra theo thứ tự:", [stack.pop() for _ in range(3)])
stack = [10]
sub = [1, 2, 3]
stack.extend(reversed(sub))        # thay "danh sách con" bằng các phần tử của nó, đảo ngược
print(stack, "-> đỉnh là", stack[-1])

lấy ra theo thứ tự: [3, 2, 1]
[10, 3, 2, 1] -> đỉnh là 1


### 3.4. Độ sâu lồng nhau và giới hạn đệ quy

Danh sách có thể lồng **rất sâu**; đệ quy sâu `d` tầng cần `d` khung gọi; Python mặc định cho ~1000 khung. Cách dùng **ngăn xếp tường minh** không bị giới hạn này (chỉ bị bộ nhớ).
#### Code demo

In [5]:
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())

giới hạn đệ quy mặc định: 1000


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

`[[1,1],2,[1,1]]`. Hình dung ngăn xếp (đỉnh bên phải), ban đầu chứa các phần tử cấp 1 **đảo ngược**: `[ [1,1] , 2 , [1,1] ]` đảo → `[[1,1]ᵇ, 2, [1,1]ᵃ]` (đỉnh là `[1,1]ᵃ`, danh sách đầu tiên).

1. `hasNext()`: đỉnh là danh sách `[1,1]ᵃ` (không phải số) ⇒ **mở** nó: pop, đẩy `1, 1` đảo ngược ⇒ đỉnh là số `1` ⇒ trả `True`.
2. `next()`: pop số `1` ⇒ trả `1`.
3. `hasNext()`: đỉnh là số `1` ⇒ `True`. `next()` ⇒ `1`.
4. `hasNext()`: đỉnh là `2` ⇒ `True`; `next()` ⇒ `2`.
5. `hasNext()`: đỉnh là danh sách `[1,1]ᵇ` ⇒ mở ⇒ `1` ... cứ thế đến `1, 1`.
6. `hasNext()`: ngăn xếp rỗng ⇒ `False`.

Kết quả `[1,1,2,1,1]`. Chú ý: **danh sách rỗng** (`[]`) khi mở sẽ không đẩy gì thêm, và vòng lặp "mở đến khi đỉnh là số" phải tiếp tục (đây là chỗ nhiều lời giải sai).

## 5. Cách 1 — Brute Force / baseline: dàn phẳng ngay trong constructor bằng đệ quy

**Ý tưởng.** Trong `__init__`, duyệt đệ quy toàn bộ cấu trúc, chép các số nguyên vào một danh sách phẳng `self.flat`; giữ chỉ số `self.i`. `hasNext()` kiểm tra `i < len(flat)`; `next()` trả `flat[i]` rồi tăng `i`.
#### Code demo

In [6]:
class NestedIteratorEager:
    def __init__(self, nestedList):
        self.flat = []
        self.i = 0

        def dfs(lst):
            for ni in lst:
                if ni.isInteger():
                    self.flat.append(ni.getInteger())
                else:
                    dfs(ni.getList())

        dfs(nestedList)

    def next(self):
        v = self.flat[self.i]
        self.i += 1
        return v

    def hasNext(self):
        return self.i < len(self.flat)


def collect(cls, nested_py):
    """Đúng như giả mã của đề."""
    it = cls(to_nested_list(nested_py))
    res = []
    while it.hasNext():
        res.append(it.next())
    return res


for nested, expected in examples:
    print(nested, "->", collect(NestedIteratorEager, nested), "| kỳ vọng:", expected)

[[1, 1], 2, [1, 1]] -> [1, 1, 2, 1, 1] | kỳ vọng: [1, 1, 2, 1, 1]
[1, [4, [6]]] -> [1, 4, 6] | kỳ vọng: [1, 4, 6]


**Độ phức tạp (Cách 1).** Gọi `N` là số số nguyên, `L` là số danh sách (kể cả rỗng), `d` là độ sâu lồng. **Constructor `O(N + L)`**, **`next`/`hasNext` `O(1)`**, **Space `O(N)`** (mảng phẳng) cộng ngăn xếp đệ quy `O(d)`.

**Nút thắt.** Phải xử lý **toàn bộ** cấu trúc ngay khi khởi tạo dù người dùng có thể chỉ cần vài phần tử đầu; dùng bộ nhớ `O(N)`; đệ quy sâu có thể tràn.

## 6. Ý tưởng thuật toán / bước đột phá

**Iterator lười với ngăn xếp.** Giữ ngăn xếp các `NestedInteger` chưa xử lý, **đảo ngược** để đỉnh là phần tử kế tiếp. Chỉ khi hỏi `hasNext()` mới "mở" vừa đủ:

```
hasNext():
    trong khi ngăn xếp không rỗng và đỉnh KHÔNG phải số nguyên:
        pop đỉnh, đẩy các phần tử của danh sách đó (đảo ngược) lên ngăn xếp
    return ngăn xếp không rỗng
next():
    hasNext()            # bảo đảm đỉnh là số nguyên
    return pop().getInteger()
```

Mỗi `NestedInteger` được đẩy và lấy ra **đúng một lần** nên tổng chi phí `O(N + L)`, **chia đều** `O(1)` amortized mỗi lần gọi; constructor chỉ đảo ngược danh sách cấp 1.

## 7. Các cách tiếp theo

### Cách 2 — Dàn phẳng sớm bằng ngăn xếp (không đệ quy)

**Giải thích.** Giống Cách 1 (xử lý hết trong constructor) nhưng **thay đệ quy bằng ngăn xếp tường minh**: đẩy danh sách cấp 1 đảo ngược; lặp pop: nếu là số thì ghi vào `flat`, nếu là danh sách thì đẩy các phần tử (đảo ngược). Không giới hạn độ sâu.
#### Code demo

In [7]:
class NestedIteratorEagerStack:
    def __init__(self, nestedList):
        self.flat = []
        self.i = 0
        stack = nestedList[::-1]
        while stack:
            ni = stack.pop()
            if ni.isInteger():
                self.flat.append(ni.getInteger())
            else:
                stack.extend(reversed(ni.getList()))

    def next(self):
        v = self.flat[self.i]
        self.i += 1
        return v

    def hasNext(self):
        return self.i < len(self.flat)


for nested, expected in examples:
    print(nested, "->", collect(NestedIteratorEagerStack, nested), "| kỳ vọng:", expected)

[[1, 1], 2, [1, 1]] -> [1, 1, 2, 1, 1] | kỳ vọng: [1, 1, 2, 1, 1]
[1, [4, [6]]] -> [1, 4, 6] | kỳ vọng: [1, 4, 6]


**Độ phức tạp Cách 2:** **Constructor `O(N + L)`**, **`next`/`hasNext` `O(1)`**, **Space `O(N)`** + ngăn xếp.

### Cách 3 — Iterator lười với ngăn xếp các `NestedInteger` (cách nộp chuẩn)

**Giải thích.** Như mục 6.
#### Code demo

In [8]:
class NestedIteratorLazyStack:
    def __init__(self, nestedList):
        self.stack = nestedList[::-1]                       # đỉnh = phần tử đầu tiên

    def next(self):
        self.hasNext()                                      # bảo đảm đỉnh là số nguyên (không hại nếu đã gọi)
        return self.stack.pop().getInteger()

    def hasNext(self):
        while self.stack and not self.stack[-1].isInteger():
            lst = self.stack.pop().getList()                # mở danh sách ở đỉnh
            self.stack.extend(reversed(lst))                # đảo ngược để phần tử đầu ở đỉnh
        return bool(self.stack)


for nested, expected in examples:
    print(nested, "->", collect(NestedIteratorLazyStack, nested), "| kỳ vọng:", expected)

[[1, 1], 2, [1, 1]] -> [1, 1, 2, 1, 1] | kỳ vọng: [1, 1, 2, 1, 1]
[1, [4, [6]]] -> [1, 4, 6] | kỳ vọng: [1, 4, 6]


**Độ phức tạp Cách 3:** **Constructor `O(n₁)`** (`n₁` = số phần tử cấp 1, chỉ đảo ngược), **`hasNext`/`next` `O(1)` amortized** (tổng `O(N + L)` cho cả quá trình), **Space `O(W)`** với `W` là tổng số phần tử "đang chờ" trên ngăn xếp (tối đa `O(N + L)` trong trường hợp xấu nhưng chỉ phát sinh khi cần).

### Cách 4 — Ngăn xếp các iterator con (bộ nhớ theo độ sâu)

**Giải thích.** Thay vì đẩy **mọi phần tử** của một danh sách con lên ngăn xếp, đẩy **một iterator** của danh sách con. Ngăn xếp chỉ lớn bằng **độ sâu** `d`. `hasNext()` lấy phần tử kế tiếp từ iterator ở đỉnh; nếu là số thì **cất vào bộ nhớ nhìn trước** `self.peeked` (để `hasNext` không tiêu thụ), nếu là danh sách thì đẩy iterator của nó, nếu iterator cạn thì pop.
#### Code demo

In [9]:
class NestedIteratorIterStack:
    def __init__(self, nestedList):
        self.stack = [iter(nestedList)]
        self.has_peek = False
        self.peeked = None

    def hasNext(self):
        if self.has_peek:
            return True
        while self.stack:
            try:
                ni = next(self.stack[-1])
            except StopIteration:
                self.stack.pop()                          # iterator cạn -> quay lại danh sách cha
                continue
            if ni.isInteger():
                self.peeked = ni.getInteger()
                self.has_peek = True
                return True
            self.stack.append(iter(ni.getList()))         # đi sâu vào danh sách con
        return False

    def next(self):
        if not self.hasNext():
            raise StopIteration
        self.has_peek = False
        return self.peeked


for nested, expected in examples:
    print(nested, "->", collect(NestedIteratorIterStack, nested), "| kỳ vọng:", expected)

[[1, 1], 2, [1, 1]] -> [1, 1, 2, 1, 1] | kỳ vọng: [1, 1, 2, 1, 1]
[1, [4, [6]]] -> [1, 4, 6] | kỳ vọng: [1, 4, 6]


**Độ phức tạp Cách 4:** **Constructor `O(1)`**, **`hasNext`/`next` `O(1)` amortized**, **Space `O(d)`** (ngăn xếp iterator, `d` = độ sâu lồng) — **tiết kiệm bộ nhớ nhất** khi danh sách rộng nhưng không quá sâu.

### Cách 5 — Generator `yield from` + nhìn trước

**Giải thích.** Viết hàm sinh đệ quy `walk(lst)` dùng `yield` cho số và `yield from walk(...)` cho danh sách con; iterator giữ generator và một ô nhìn trước để `hasNext()` không tiêu thụ. Ngắn gọn, đúng kiểu Python, nhưng đệ quy theo độ sâu.
#### Code demo

In [10]:
class NestedIteratorGenerator:
    def __init__(self, nestedList):
        def walk(lst):
            for ni in lst:
                if ni.isInteger():
                    yield ni.getInteger()
                else:
                    yield from walk(ni.getList())

        self.gen = walk(nestedList)
        self.has_peek = False
        self.peeked = None

    def hasNext(self):
        if not self.has_peek:
            try:
                self.peeked = next(self.gen)
                self.has_peek = True
            except StopIteration:
                return False
        return True

    def next(self):
        if not self.hasNext():
            raise StopIteration
        self.has_peek = False
        return self.peeked


for nested, expected in examples:
    print(nested, "->", collect(NestedIteratorGenerator, nested), "| kỳ vọng:", expected)

[[1, 1], 2, [1, 1]] -> [1, 1, 2, 1, 1] | kỳ vọng: [1, 1, 2, 1, 1]
[1, [4, [6]]] -> [1, 4, 6] | kỳ vọng: [1, 4, 6]


**Độ phức tạp Cách 5:** **Constructor `O(1)`** (generator chưa chạy), **`hasNext`/`next` `O(1)` amortized**, **Space `O(d)`** (các generator lồng nhau; `yield from` làm chuỗi gọi dài `d` mỗi lần lấy phần tử sâu).

> Cách 3 là bản nộp thông dụng và dễ chứng minh. Cách 4 tiết kiệm bộ nhớ nhất. Cách 1/2 đơn giản nhưng dàn phẳng ngay (không lười).

## 8. TRACE từng bước

Ta trace Cách 3 trên `[[1,1],2,[1,1]]`: in ngăn xếp (đỉnh bên phải) trước và sau mỗi lần gọi `hasNext()` / `next()`.
#### Code demo

In [11]:
def trace_lazy(nested_py):
    ni = to_nested_list(nested_py)
    stack = ni[::-1]
    print(f"nestedList = {nested_py}")
    print(f"  khởi tạo: ngăn xếp (đỉnh bên phải) = {stack}")
    out = []
    step = 0
    while True:
        step += 1
        opened = []
        while stack and not stack[-1].isInteger():
            lst = stack.pop().getList()
            opened.append(lst)
            stack.extend(reversed(lst))
        has = bool(stack)
        print(f"  hasNext #{step}: {'mở ' + str(opened) if opened else 'không cần mở'} -> ngăn xếp = {stack} -> {has}")
        if not has:
            break
        v = stack.pop().getInteger()
        out.append(v)
        print(f"  next    #{step}: trả {v} -> ngăn xếp = {stack}")
    print("  kết quả:", out)
    return out


trace_lazy([[1, 1], 2, [1, 1]])
print()
trace_lazy([[], [[]], 3, []])

nestedList = [[1, 1], 2, [1, 1]]
  khởi tạo: ngăn xếp (đỉnh bên phải) = [[1, 1], 2, [1, 1]]
  hasNext #1: mở [[1, 1]] -> ngăn xếp = [[1, 1], 2, 1, 1] -> True
  next    #1: trả 1 -> ngăn xếp = [[1, 1], 2, 1]
  hasNext #2: không cần mở -> ngăn xếp = [[1, 1], 2, 1] -> True
  next    #2: trả 1 -> ngăn xếp = [[1, 1], 2]
  hasNext #3: không cần mở -> ngăn xếp = [[1, 1], 2] -> True
  next    #3: trả 2 -> ngăn xếp = [[1, 1]]
  hasNext #4: mở [[1, 1]] -> ngăn xếp = [1, 1] -> True
  next    #4: trả 1 -> ngăn xếp = [1]
  hasNext #5: không cần mở -> ngăn xếp = [1] -> True
  next    #5: trả 1 -> ngăn xếp = []
  hasNext #6: không cần mở -> ngăn xếp = [] -> False
  kết quả: [1, 1, 2, 1, 1]

nestedList = [[], [[]], 3, []]
  khởi tạo: ngăn xếp (đỉnh bên phải) = [[], 3, [[]], []]
  hasNext #1: mở [[], [[]], []] -> ngăn xếp = [[], 3] -> True
  next    #1: trả 3 -> ngăn xếp = [[]]
  hasNext #2: mở [[]] -> ngăn xếp = [] -> False
  kết quả: [3]


[3]

Bảng tóm tắt (`[1,[4,[6]]]`):

| gọi | ngăn xếp trước (đỉnh bên phải) | thao tác | trả về |
|---|---|---|---|
| khởi tạo | `[[4,[6]], 1]` | đảo ngược cấp 1 | |
| `hasNext` | đỉnh là số `1` | không mở | `True` |
| `next` | `[[4,[6]], 1]` | pop `1` | `1` |
| `hasNext` | đỉnh là danh sách `[4,[6]]` | mở: đẩy `[[6], 4]` ⇒ đỉnh `4` | `True` |
| `next` | `[[6], 4]` | pop `4` | `4` |
| `hasNext` | đỉnh là danh sách `[6]` | mở: đẩy `6` ⇒ đỉnh `6` | `True` |
| `next` | `[6]` | pop `6` | `6` |
| `hasNext` | ngăn xếp rỗng | | `False` |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3. Trên LeetCode `NestedInteger` đã có sẵn; ta không định nghĩa lại. Đúng `class NestedIterator`, đúng ba phương thức `__init__(self, nestedList)`, `next(self) -> int`, `hasNext(self) -> bool`.
#### Code demo

In [12]:
class NestedIterator:
    def __init__(self, nestedList: [NestedInteger]):
        self.stack = nestedList[::-1]                      # đỉnh ngăn xếp = phần tử đầu tiên

    def next(self) -> int:
        self.hasNext()                                     # bảo đảm đỉnh là một số nguyên
        return self.stack.pop().getInteger()

    def hasNext(self) -> bool:
        while self.stack and not self.stack[-1].isInteger():
            lst = self.stack.pop().getList()               # mở danh sách ở đỉnh (kể cả danh sách rỗng)
            self.stack.extend(reversed(lst))               # đảo ngược để giữ thứ tự trái -> phải
        return bool(self.stack)


# Your NestedIterator object will be instantiated and called as such:
# i, v = NestedIterator(nestedList), []
# while i.hasNext(): v.append(i.next())

for nested, expected in examples:
    assert collect(NestedIterator, nested) == expected, (nested, expected)
print("OK: 2 ví dụ của đề")

OK: 2 ví dụ của đề


## 10. Vì sao thuật toán đúng

Gọi dãy dàn phẳng đúng là `F` (thứ tự DFS trái → phải của các số nguyên).

**Bất biến của Cách 3.** Nối các phần tử trên ngăn xếp **từ đỉnh xuống đáy**, rồi dàn phẳng từng phần tử theo thứ tự đó, ta được **đúng phần chưa trả** của `F`. (Ngăn xếp là "phần còn lại của F" dưới dạng đã mở một phần.)

- **Khởi tạo:** ngăn xếp = cấp 1 đảo ngược ⇒ từ đỉnh xuống đáy chính là cấp 1 theo thứ tự trái → phải ⇒ dàn phẳng = `F`.
- **Mở một danh sách ở đỉnh:** thay `[lst]` bằng các phần tử của `lst` (đảo ngược để đỉnh là phần tử đầu của `lst`) **không đổi** dãy dàn phẳng (vì dàn phẳng của `lst` là nối các dàn phẳng các phần tử của nó). Với `lst` rỗng, ngăn xếp chỉ **mất** một phần tử mà dàn phẳng không đổi.
- **`hasNext`:** lặp mở đến khi đỉnh là số **hoặc** ngăn xếp rỗng. Vì mỗi lần mở làm giảm số "danh sách chưa mở", sau hữu hạn bước dừng. Nếu rỗng ⇒ phần chưa trả rỗng ⇒ `False`; ngược lại đỉnh là phần tử đầu tiên chưa trả ⇒ `True`.
- **`next`:** sau `hasNext()` đỉnh là số nguyên và chính là phần tử đầu của phần chưa trả ⇒ pop và trả đúng giá trị; bất biến được giữ.
- **`hasNext` không tiêu thụ:** nó chỉ **mở** (không đổi dãy dàn phẳng) chứ không pop số nguyên, nên gọi nhiều lần không mất phần tử.
- **Cách 4:** mỗi iterator trên ngăn xếp tương ứng "phần còn lại của một danh sách đang mở"; `peeked` chứa phần tử đầu của phần chưa trả. **Cách 1/2/5:** dựa trực tiếp vào định nghĩa DFS.

## 11. Phân tích Time / Space Complexity

Gọi `N` là số số nguyên, `L` là số danh sách (kể cả rỗng), `d` là độ sâu lồng, `n₁` là số phần tử cấp 1 (≤ 500).

| Cách | Constructor | `next` / `hasNext` | Space |
|---|---|---|---|
| 1. Dàn phẳng sớm (đệ quy) | `O(N + L)` | `O(1)` | `O(N)` + `O(d)` ngăn xếp đệ quy |
| 2. Dàn phẳng sớm (ngăn xếp) | `O(N + L)` | `O(1)` | `O(N)` |
| 3. Lười, ngăn xếp `NestedInteger` | `O(n₁)` | `O(1)` amortized | `O(W)` ≤ `O(N + L)` |
| 4. Lười, ngăn xếp iterator | `O(1)` | `O(1)` amortized | `O(d)` |
| 5. Generator + nhìn trước | `O(1)` | `O(1)` amortized | `O(d)` |

Tổng chi phí duyệt hết bằng mọi cách đều `O(N + L)`. Khác biệt là **khi nào** làm việc (sớm hay lười) và **bộ nhớ**.

## 12. Bộ test

Khung test:

- `collect(cls, nested)`: đúng giả mã của đề (hỏi `hasNext()` rồi `next()`).
- `drive(cls, nested, plan)`: **gọi xen kẽ** theo kế hoạch `plan` (chuỗi các ký tự `h` = `hasNext()`, `n` = `next()`), trả danh sách kết quả; kiểm tra **`hasNext` không tiêu thụ**.
- `expected_flatten(nested)`: oracle độc lập bằng regex trên chuỗi biểu diễn.
#### Code demo

In [13]:
def drive(cls, nested_py, plan):
    it = cls(to_nested_list(nested_py))
    out = []
    for ch in plan:
        out.append(it.hasNext() if ch == "h" else it.next())
    return out


impls = {
    "eager": NestedIteratorEager,
    "eager_stack": NestedIteratorEagerStack,
    "lazy_stack": NestedIteratorLazyStack,
    "iter_stack": NestedIteratorIterStack,
    "generator": NestedIteratorGenerator,
    "submit": NestedIterator,
}

for nested, expected in examples:
    assert expected_flatten(nested) == expected
    for key, cls in impls.items():
        assert collect(cls, nested) == expected, (key, nested, expected, collect(cls, nested))
print("PASS ví dụ của đề (cả oracle regex)")

PASS ví dụ của đề (cả oracle regex)


### 12.1. Các trường hợp biên
#### Code demo

In [14]:
edge_cases = [
    ([5], [5], "một số nguyên"),
    ([[]], [], "một danh sách rỗng"),
    ([[], []], [], "hai danh sách rỗng"),
    ([[[]]], [], "rỗng lồng sâu"),
    ([[], 1, []], [1], "số nằm giữa các danh sách rỗng"),
    ([[], [[]], 3, []], [3], "rỗng lồng nhiều chỗ, một số ở giữa"),
    ([[[[1]]]], [1], "số nằm sâu 4 tầng"),
    ([1, 2, 3], [1, 2, 3], "danh sách phẳng"),
    ([[1], [2], [3]], [1, 2, 3], "mỗi phần tử là danh sách một số"),
    ([-1000000, [0, [1000000]]], [-1000000, 0, 1000000], "giá trị biên và 0, số âm"),
    ([[1, [2, [3, [4]]]], 5], [1, 2, 3, 4, 5], "lồng dần rồi số ở ngoài"),
    ([[], [], [1, []], [[2], []], []], [1, 2], "rất nhiều danh sách rỗng"),
]
for nested, expected, label in edge_cases:
    assert expected_flatten(nested) == expected, (label, expected_flatten(nested))
    for key, cls in impls.items():
        assert collect(cls, nested) == expected, (key, label, nested, expected, collect(cls, nested))
    print("PASS", label)

PASS một số nguyên
PASS một danh sách rỗng
PASS hai danh sách rỗng
PASS rỗng lồng sâu
PASS số nằm giữa các danh sách rỗng
PASS rỗng lồng nhiều chỗ, một số ở giữa
PASS số nằm sâu 4 tầng
PASS danh sách phẳng
PASS mỗi phần tử là danh sách một số
PASS giá trị biên và 0, số âm
PASS lồng dần rồi số ở ngoài
PASS rất nhiều danh sách rỗng


### 12.2. Tính chất và phản ví dụ

- `hasNext()` **không tiêu thụ**: gọi nhiều lần vẫn cùng kết quả và không mất phần tử.
- Sau khi hết phần tử, `hasNext()` luôn `False`.
- `next()` có thể gọi **mà không cần** `hasNext()` trước (nếu còn phần tử).
- **Phản ví dụ** cho `hasNext` chỉ kiểm tra ngăn xếp khác rỗng: `[[]]` (xem Lỗi 1).
#### Code demo

In [15]:
def random_nested(rng, max_len, depth, p_list=0.35, lo=-10 ** 6, hi=10 ** 6):
    """Danh sách lồng nhau ngẫu nhiên (cho phép danh sách rỗng)."""
    out = []
    for _ in range(rng.randint(0, max_len)):
        if depth > 0 and rng.random() < p_list:
            out.append(random_nested(rng, max_len, depth - 1, p_list, lo, hi))
        else:
            out.append(rng.randint(lo, hi))
    return out


def random_plan(rng, count):
    """Kế hoạch gọi: trước mỗi next có 0..2 lần hasNext; cuối cùng vài hasNext."""
    plan = []
    for _ in range(count):
        plan.append("h" * rng.randint(0, 2) + "n")
    return "".join(plan) + "hh"


rng = random.Random(341)
for _ in range(500):
    nested = random_nested(rng, 4, 3)
    expected = expected_flatten(nested)
    plan = random_plan(rng, len(expected))
    for key, cls in impls.items():
        res = drive(cls, nested, plan)
        got = []
        it_check = []
        for ch, r in zip(plan, res):
            if ch == "n":
                got.append(r)
            else:
                it_check.append(r)
        assert got == expected, (key, nested, plan, got, expected)
        assert it_check[-1] is False and it_check[-2] is False        # hết phần tử: hasNext luôn False
        # hasNext trước mỗi next phải là True nếu còn phần tử
        pos = 0
        for ch, r in zip(plan, res):
            if ch == "h":
                assert r == (pos < len(expected)), (key, nested, plan)
            else:
                pos += 1
print("PASS: gọi xen kẽ hasNext/next ngẫu nhiên; hasNext không tiêu thụ và đúng True/False")

PASS: gọi xen kẽ hasNext/next ngẫu nhiên; hasNext không tiêu thụ và đúng True/False


### 12.3. Kiểm tra ngẫu nhiên (có seed) và dữ liệu lớn
#### Code demo

In [16]:
rng = random.Random(2341)
for trial in range(3000):
    nested = random_nested(rng, rng.randint(1, 6), rng.randint(0, 5), rng.choice([0.2, 0.4, 0.6]))
    if not nested:
        nested = [rng.randint(-5, 5)]
    expected = expected_flatten(nested)
    for key, cls in impls.items():
        assert collect(cls, nested) == expected, (key, nested)
print("PASS 3000 danh sách lồng nhau ngẫu nhiên (khung collect) so với oracle regex")

# Dữ liệu lớn: 500 phần tử cấp 1, mỗi phần tử là cấu trúc lớn
rng = random.Random(7)
deep200 = 1
for _ in range(200):
    deep200 = [deep200]
big_cases = [
    ("phẳng 500 số", [rng.randint(-10 ** 6, 10 ** 6) for _ in range(500)]),
    ("500 danh sách, mỗi danh sách 40 số", [[rng.randint(-10 ** 6, 10 ** 6) for _ in range(40)] for _ in range(500)]),
    ("lồng ngẫu nhiên rộng", [random_nested(rng, 6, 5, 0.5) for _ in range(500)]),
    ("500 danh sách rỗng lồng đôi", [[[]] for _ in range(500)]),
    ("lồng sâu 200 tầng + 499 nhánh nông", [deep200] + [[[[rng.randint(0, 9)]]] for _ in range(499)]),
]
for label, nested in big_cases:
    expected = expected_flatten(nested)
    row = []
    for key, cls in impls.items():
        t0 = time.perf_counter()
        res = collect(cls, nested)
        dt = (time.perf_counter() - t0) * 1000
        assert res == expected, (label, key)
        row.append(f"{key}={dt:6.1f}ms")
    print(f"{label:<38} N={len(expected):<6} | " + " ".join(row))
print("PASS dữ liệu lớn")


# Tính "lười": đếm số lần gọi giao diện NestedInteger ngay trong constructor
big = [[rng.randint(0, 9) for _ in range(40)] for _ in range(500)]
ni_list = to_nested_list(big)
print("\nSố lần gọi giao diện NestedInteger trong constructor (500 danh sách x 40 số):")
for key, cls in impls.items():
    NestedInteger.calls = 0
    cls(ni_list)
    print(f"  {key:<12} {NestedInteger.calls:>6} lần")
    if key in ("lazy_stack", "iter_stack", "generator", "submit"):
        assert NestedInteger.calls == 0, key                 # lười: chưa đụng tới cấu trúc
    else:
        assert NestedInteger.calls > 20_000, key             # sớm: duyệt hết ngay

PASS 3000 danh sách lồng nhau ngẫu nhiên (khung collect) so với oracle regex


phẳng 500 số                           N=500    | eager=   0.4ms eager_stack=   0.4ms lazy_stack=   0.5ms iter_stack=   0.3ms generator=   0.3ms submit=   0.4ms


500 danh sách, mỗi danh sách 40 số     N=20000  | eager=  15.2ms eager_stack=  12.9ms lazy_stack=  15.5ms iter_stack=  13.3ms generator=  12.2ms submit=  15.5ms


lồng ngẫu nhiên rộng                   N=21597  | eager=  43.0ms eager_stack=  25.1ms lazy_stack=  26.9ms iter_stack=  26.1ms generator=  26.1ms submit=  28.0ms
500 danh sách rỗng lồng đôi            N=0      | eager=   0.6ms eager_stack=   0.6ms lazy_stack=   0.6ms iter_stack=   0.7ms generator=   0.6ms submit=   0.6ms
lồng sâu 200 tầng + 499 nhánh nông     N=500    | eager=   1.6ms eager_stack=   1.4ms lazy_stack=   1.5ms iter_stack=   1.6ms generator=   1.4ms submit=   1.5ms
PASS dữ liệu lớn



Số lần gọi giao diện NestedInteger trong constructor (500 danh sách x 40 số):
  eager         41000 lần
  eager_stack   41000 lần
  lazy_stack        0 lần
  iter_stack        0 lần
  generator         0 lần
  submit            0 lần


**Nhận xét.** Các cách **lười** (`lazy_stack`, `iter_stack`, `generator`) không gọi giao diện nào trong constructor (chỉ đảo ngược/bọc danh sách cấp 1), còn các cách **sớm** (`eager`, `eager_stack`) duyệt toàn bộ ngay lập tức. Nếu người dùng chỉ cần vài phần tử đầu, cách lười tiết kiệm rất nhiều.

## 13. Các lỗi người mới hay gặp

### Lỗi 1: `hasNext()` chỉ kiểm tra ngăn xếp khác rỗng
#### Code sai

In [17]:
class SaiHasNextChiKiemTraRong:
    def __init__(self, nestedList):
        self.stack = nestedList[::-1]

    def hasNext(self):
        return len(self.stack) > 0                         # SAI: đỉnh có thể là danh sách (kể cả rỗng)

    def next(self):
        while self.stack and not self.stack[-1].isInteger():
            self.stack.extend(reversed(self.stack.pop().getList()))
        return self.stack.pop().getInteger()


for nested in ([[]], [[], []], [[], 1]):
    it = SaiHasNextChiKiemTraRong(to_nested_list(nested))
    res = []
    try:
        while it.hasNext():
            res.append(it.next())
        print(nested, "sai ->", res)
    except IndexError as e:
        print(nested, "sai -> IndexError:", e, "| đúng:", collect(NestedIterator, nested))

[[]] sai -> IndexError: pop from empty list | đúng: []
[[], []] sai -> IndexError: pop from empty list | đúng: []
[[], 1] sai -> [1]


**Vì sao sai:** với `[[]]`, ngăn xếp có một phần tử (danh sách rỗng) nên `hasNext()` trả `True` nhưng thực ra **không còn số nguyên** ⇒ `next()` lỗi. `hasNext` phải **mở** các danh sách cho tới khi đỉnh là số hoặc ngăn xếp rỗng. **Code đúng:** vòng lặp mở trong `hasNext`.

### Lỗi 2: Đẩy danh sách con lên ngăn xếp mà không đảo ngược
#### Code sai

In [18]:
class SaiKhongDaoNguoc:
    def __init__(self, nestedList):
        self.stack = nestedList[::-1]

    def hasNext(self):
        while self.stack and not self.stack[-1].isInteger():
            self.stack.extend(self.stack.pop().getList())      # SAI: thiếu reversed(...)
        return bool(self.stack)

    def next(self):
        self.hasNext()
        return self.stack.pop().getInteger()


nested = [[1, 2], 3, [4, 5]]
print("sai  :", collect(SaiKhongDaoNguoc, nested), "| đúng:", collect(NestedIterator, nested))
assert collect(SaiKhongDaoNguoc, nested) != collect(NestedIterator, nested)

sai  : [2, 1, 3, 5, 4] | đúng: [1, 2, 3, 4, 5]


**Vì sao sai:** ngăn xếp lấy phần tử **vào sau cùng**; mở `[1, 2]` bằng `extend` làm `2` nằm trên đỉnh, nên ra `2, 1`. Phải `extend(reversed(...))` để phần tử đầu ở đỉnh. **Code đúng:** `self.stack.extend(reversed(lst))`.

### Lỗi 3: `hasNext()` làm mất phần tử (tiêu thụ số nguyên)
#### Code sai

In [19]:
class SaiHasNextTieuThu:
    def __init__(self, nestedList):
        self.stack = nestedList[::-1]
        self.pending = None

    def hasNext(self):
        while self.stack:
            top = self.stack.pop()                          # SAI: lấy luôn số nguyên ra khỏi ngăn xếp
            if top.isInteger():
                self.pending = top.getInteger()
                return True
            self.stack.extend(reversed(top.getList()))
        return False

    def next(self):
        return self.pending


nested = [1, 2, 3]
print("kế hoạch 'hhnhnhn' (gọi hasNext hai lần liền):")
print("  sai  :", [r for r in drive(SaiHasNextTieuThu, nested, "hhnhnhn") if not isinstance(r, bool)])
print("  đúng :", [r for r in drive(NestedIterator, nested, "hhnhnhn") if not isinstance(r, bool)])
assert [r for r in drive(SaiHasNextTieuThu, nested, "hhnhnhn") if not isinstance(r, bool)] != [1, 2, 3]
assert collect(SaiHasNextTieuThu, nested) == [1, 2, 3]        # nhưng đúng khi chỉ dùng mẫu hasNext-next xen kẽ

kế hoạch 'hhnhnhn' (gọi hasNext hai lần liền):
  sai  : [2, 3, 3]
  đúng : [1, 2, 3]


**Vì sao sai:** mỗi lần gọi `hasNext()` lại pop một số nguyên nữa, nên gọi hai lần liền làm **mất** một phần tử. Khung test của đề luôn xen kẽ `hasNext`/`next` nên lỗi này có thể lọt, nhưng hợp đồng iterator yêu cầu `hasNext` **không tiêu thụ**. **Code đúng:** chỉ mở danh sách; không pop số nguyên (hoặc cất vào ô nhìn trước và không pop lần nữa nếu đã có).

### Lỗi 4: Chỉ dàn phẳng một tầng
#### Code sai

In [20]:
def sai_dan_phang_mot_tang(nestedList):
    flat = []
    for ni in nestedList:
        if ni.isInteger():
            flat.append(ni.getInteger())
        else:
            for sub in ni.getList():
                flat.append(sub.getInteger())                 # SAI: sub vẫn có thể là danh sách
    return flat


nested = [1, [4, [6]]]
print("sai  :", sai_dan_phang_mot_tang(to_nested_list(nested)), "| đúng:", collect(NestedIterator, nested))
assert sai_dan_phang_mot_tang(to_nested_list(nested)) != collect(NestedIterator, nested)

sai  : [1, 4, None] | đúng: [1, 4, 6]


**Vì sao sai:** lồng nhau có thể sâu nhiều tầng; gặp `[6]` ở tầng sâu hơn, `getInteger()` không có nghĩa (trả `None`). **Code đúng:** đệ quy/ngăn xếp cho **mọi** tầng.

### Lỗi 5: Đệ quy quá sâu khi dàn phẳng
#### Code sai

In [21]:
def build_deep(depth):
    node = NestedInteger(7)
    for _ in range(depth):
        node = NestedInteger([node])                           # dựng lặp để tránh chính việc dựng bị đệ quy
    return [node]


deep = build_deep(3000)


def thu(cls):
    try:
        it = cls(deep)
        res = []
        while it.hasNext():
            res.append(it.next())
        return res
    except RecursionError:
        return "RecursionError"


for key in ("eager", "generator", "eager_stack", "lazy_stack", "iter_stack"):
    print(f"{key:<12} độ sâu 3000 ->", thu(impls[key]))
assert thu(NestedIteratorEager) == "RecursionError"
assert thu(NestedIteratorLazyStack) == [7] and thu(NestedIteratorIterStack) == [7] and thu(NestedIteratorEagerStack) == [7]

eager        độ sâu 3000 -> RecursionError
generator    độ sâu 3000 -> RecursionError
eager_stack  độ sâu 3000 -> [7]
lazy_stack   độ sâu 3000 -> [7]
iter_stack   độ sâu 3000 -> [7]


**Vì sao nguy hiểm:** các cách dùng **đệ quy** (`eager`, `generator`) tràn ngăn xếp khi cấu trúc sâu hơn giới hạn đệ quy của Python (~1000). Các cách dùng **ngăn xếp tường minh** vẫn đúng. (Đề của LeetCode không có cấu trúc quá sâu, nhưng đây là điểm nên biết.) **Code đúng:** ngăn xếp tường minh (Cách 2/3/4).

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**iterator trên cấu trúc lồng nhau**" → **ngăn xếp**; đẩy **đảo ngược** để giữ thứ tự trái → phải.
- Nhớ nguyên tắc: **`hasNext()` làm việc** (mở danh sách đến khi đỉnh là số), **`next()` chỉ gọi `hasNext()` rồi pop**.
- **Danh sách rỗng** (`[]`, `[[]]`, ...) là ca biên kinh điển: `hasNext` phải dùng **vòng lặp `while`**, không `if`.
- Muốn tiết kiệm bộ nhớ: **ngăn xếp iterator** (`O(d)`); muốn code ngắn: **generator** hoặc **dàn phẳng sớm** (nhưng không lười).
- `hasNext` **không được tiêu thụ** phần tử (dùng ô nhìn trước nếu cần).
- Họ hàng: **Flatten 2D Vector** (251), **Peeking Iterator** (284), **Zigzag Iterator** (281), **Nested List Weight Sum** (339, 364), **Mini Parser** (385), **Binary Search Tree Iterator** (173), **Flatten Binary Tree to Linked List** (114).
- Test thiết kế: khung `collect` + gọi xen kẽ ngẫu nhiên + oracle độc lập (regex).
- Thử: `[[]]`, `[[], 1, []]`, `[[[[1]]]]`, `hasNext()` nhiều lần liền.

## 15. Biến thể của bài

1. **Nested List Weight Sum** (339, Premium): tổng `giá trị × độ sâu`.
2. **Nested List Weight Sum II** (364, Premium): trọng số theo độ sâu ngược (lá nông có trọng số lớn).
3. **Peeking Iterator** (284): thêm `peek()` không tiêu thụ.
4. **Mini Parser** (385): phân tích chuỗi `"[123,[456,[789]]]"` thành `NestedInteger`.
5. **Zigzag Iterator** (281) / **Flatten 2D Vector** (251): iterator trên nhiều danh sách.
#### Code demo

In [22]:
def depth_sum(nestedList):
    """LeetCode 339: tổng giá trị x độ sâu (cấp 1 có độ sâu 1). Lặp bằng ngăn xếp (nút, độ sâu)."""
    total = 0
    stack = [(ni, 1) for ni in nestedList]
    while stack:
        ni, d = stack.pop()
        if ni.isInteger():
            total += ni.getInteger() * d
        else:
            stack.extend((c, d + 1) for c in ni.getList())
    return total


class PeekableNested(NestedIterator):
    """Biến thể 284: thêm peek() không tiêu thụ."""

    def peek(self):
        self.hasNext()
        return self.stack[-1].getInteger()


print(depth_sum(to_nested_list([[1, 1], 2, [1, 1]])), depth_sum(to_nested_list([1, [4, [6]]])))   # 10 và 27
p = PeekableNested(to_nested_list([[1, 2], 3]))
print(p.peek(), p.peek(), p.next(), p.peek(), p.next(), p.next(), p.hasNext())    # 1 1 1 2 2 3 False

10 27
1 1 1 2 2 3 False


## 16. So sánh các cách làm

| Cách | Ý tưởng | Constructor | `next`/`hasNext` | Space | Khi nào dùng |
|---|---|---|---|---|---|
| 1. Dàn phẳng sớm (đệ quy) | dàn phẳng ngay trong constructor | `O(N+L)` | `O(1)` | `O(N)` | baseline, dễ viết |
| 2. Dàn phẳng sớm (ngăn xếp) | như 1, không đệ quy | `O(N+L)` | `O(1)` | `O(N)` | cấu trúc sâu |
| 3. Lười, ngăn xếp `NestedInteger` | mở dần ở `hasNext` | `O(n₁)` | `O(1)` am. | `O(W)` | **bản nộp chuẩn** |
| 4. Lười, ngăn xếp iterator | iterator con + nhìn trước | `O(1)` | `O(1)` am. | `O(d)` | tiết kiệm bộ nhớ nhất |
| 5. Generator + nhìn trước | `yield from` | `O(1)` | `O(1)` am. | `O(d)` | code ngắn, kiểu Python |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán kết quả `collect` (danh sách dàn phẳng) cho: (a) `[[1,[2]],[],3]`; (b) `[[[]]]`; (c) `[[[[5]]], 6]`; (d) `[1,[2,[3,[4]]]]`.

**Bài 2 (tính bằng tay).** Với `[[1,2],[3,[4]]]`, vẽ ngăn xếp (đỉnh bên phải) sau mỗi lần `hasNext()` theo Cách 3.

**Bài 3 (điền TODO).** Hoàn thiện lớp `LazyTodo` theo Cách 3.
#### Code demo

In [23]:
class LazyTodo:
    def __init__(self, nestedList):
        pass
        # TODO 1: self.stack = nestedList đảo ngược

    def hasNext(self):
        pass
        # TODO 2: while ngăn xếp không rỗng và đỉnh không phải số nguyên: pop đỉnh, extend(reversed(getList()))
        # TODO 3: return ngăn xếp không rỗng

    def next(self):
        pass
        # TODO 4: gọi hasNext() rồi pop().getInteger()


# Tự làm xong rồi mới mở test:
# for nested, expected in examples: assert collect(LazyTodo, nested) == expected

**Bài 4 (tự viết lại).** Viết `NestedIteratorWithPeek` có thêm `peek()` (xem phần tử kế tiếp **không** lấy ra), dựa trên Cách 3. Kiểm tra: `peek()` gọi nhiều lần cho cùng một giá trị; sau `next()` thì `peek()` trả giá trị kế.

**Bài 5 (biến thể / mini challenge).** Viết `weighted_sum_reverse_depth(nestedList)` (LeetCode 364): trọng số của một số là `maxDepth - depth + 1`, với `depth` là độ sâu của nó (cấp 1 có độ sâu 1) và `maxDepth` là độ sâu lớn nhất của một **số nguyên** trong cấu trúc. Thử `[[1,1],2,[1,1]]` → `8`, `[1,[4,[6]]]` → `17`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: bỏ qua mọi `[]`, giữ thứ tự trái → phải. Bài 2: đỉnh bên phải, mở danh sách ở đỉnh. Bài 4: `peek` = `hasNext()` rồi nhìn đỉnh. Bài 5: tính độ sâu tối đa trước.
- **Mức 2 (vừa):** Bài 1: `[1,2,3]`, `[]`, `[5,6]`, `[1,2,3,4]`. Bài 2: ban đầu `[[3,[4]], [1,2]]` (đỉnh `[1,2]`); `hasNext` #1: `[[3,[4]], 2, 1]`; sau khi lấy `1, 2` thì `hasNext` #3: mở `[3,[4]]` ⇒ `[[4], 3]`; sau khi lấy `3`, `hasNext` #4: mở `[4]` ⇒ `[4]`; cuối cùng rỗng. Bài 3: khung Cách 3.
- **Mức 3 (sâu):** Bài 4: `return self.stack[-1].getInteger()` sau khi `hasNext()`. Bài 5: duyệt lặp ghi nhận `(giá trị, độ sâu)` của mọi số nguyên và `max_depth` = độ sâu lớn nhất trong số đó; kết quả `Σ giá trị × (max_depth − depth + 1)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
# Bài 1
assert [collect(NestedIterator, v) for v in ([[1, [2]], [], 3], [[[]]], [[[[5]]], 6], [1, [2, [3, [4]]]])] == [[1, 2, 3], [], [5, 6], [1, 2, 3, 4]]


# Bài 2 (kiểm tra bằng trace_lazy: đầu ra cuối là [1, 2, 3, 4])
assert trace_lazy([[1, 2], [3, [4]]]) == [1, 2, 3, 4]


# Bài 3
class LazyTodoAns:
    def __init__(self, nestedList):
        self.stack = nestedList[::-1]

    def hasNext(self):
        while self.stack and not self.stack[-1].isInteger():
            self.stack.extend(reversed(self.stack.pop().getList()))
        return bool(self.stack)

    def next(self):
        self.hasNext()
        return self.stack.pop().getInteger()


for nested, expected in examples:
    assert collect(LazyTodoAns, nested) == expected


# Bài 4
class NestedIteratorWithPeek(LazyTodoAns):
    def peek(self):
        self.hasNext()
        return self.stack[-1].getInteger()


pk = NestedIteratorWithPeek(to_nested_list([[1, 2], [], 3]))
assert pk.peek() == 1 and pk.peek() == 1 and pk.next() == 1 and pk.peek() == 2
assert pk.next() == 2 and pk.peek() == 3 and pk.next() == 3 and not pk.hasNext()


# Bài 5
def weighted_sum_reverse_depth(nestedList):
    pairs = []                                               # (giá trị, độ sâu)
    max_depth = 0
    stack = [(ni, 1) for ni in nestedList]
    while stack:
        ni, d = stack.pop()
        if ni.isInteger():
            pairs.append((ni.getInteger(), d))
            max_depth = max(max_depth, d)
        else:
            stack.extend((c, d + 1) for c in ni.getList())
    return sum(v * (max_depth - d + 1) for v, d in pairs)


assert weighted_sum_reverse_depth(to_nested_list([[1, 1], 2, [1, 1]])) == 8
assert weighted_sum_reverse_depth(to_nested_list([1, [4, [6]]])) == 17
print("PASS đáp án bài tập")

nestedList = [[1, 2], [3, [4]]]
  khởi tạo: ngăn xếp (đỉnh bên phải) = [[3, [4]], [1, 2]]
  hasNext #1: mở [[1, 2]] -> ngăn xếp = [[3, [4]], 2, 1] -> True
  next    #1: trả 1 -> ngăn xếp = [[3, [4]], 2]
  hasNext #2: không cần mở -> ngăn xếp = [[3, [4]], 2] -> True
  next    #2: trả 2 -> ngăn xếp = [[3, [4]]]
  hasNext #3: mở [[3, [4]]] -> ngăn xếp = [[4], 3] -> True
  next    #3: trả 3 -> ngăn xếp = [[4]]
  hasNext #4: mở [[4]] -> ngăn xếp = [4] -> True
  next    #4: trả 4 -> ngăn xếp = []
  hasNext #5: không cần mở -> ngăn xếp = [] -> False
  kết quả: [1, 2, 3, 4]
PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (iterator lười với ngăn xếp)

```python
class NestedIterator:
    def __init__(self, nestedList: [NestedInteger]):
        self.stack = nestedList[::-1]

    def next(self) -> int:
        self.hasNext()
        return self.stack.pop().getInteger()

    def hasNext(self) -> bool:
        while self.stack and not self.stack[-1].isInteger():
            self.stack.extend(reversed(self.stack.pop().getList()))
        return bool(self.stack)
```

**Checklist**

- [ ] Khởi tạo ngăn xếp bằng danh sách cấp 1 **đảo ngược**?
- [ ] `hasNext()` dùng **`while`** để mở danh sách đến khi đỉnh là số (xử lý được `[]`, `[[]]`)?
- [ ] Khi mở danh sách con dùng `extend(reversed(...))` (không quên đảo ngược)?
- [ ] `next()` gọi `hasNext()` trước rồi mới `pop().getInteger()`?
- [ ] `hasNext()` **không tiêu thụ** số nguyên (gọi nhiều lần vẫn đúng)?
- [ ] Không dùng đệ quy sâu (hoặc đã cân nhắc giới hạn)?
- [ ] Đã thử `[[]]`, `[[], 1, []]`, `[[[[1]]]]`, 500 phần tử cấp 1?
- [ ] Skeleton đúng `class NestedIterator` với `__init__(self, nestedList: [NestedInteger])`, `next(self) -> int`, `hasNext(self) -> bool`?